<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇮🇹 Italy Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    GTFS Source: dati.toscana.it · NeTEx Source: cciss.it (Italian NAP) · Operator: Trenitalia · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

**GTFS — Trenitalia Tuscany**
- Source: Tuscany Open Data Portal (`dati.toscana.it`)
- Dataset page: https://dati.toscana.it/dataset/rt-oraritb
- File used: `trenitalia.gtfs` (ZIP)
- Coverage: Trenitalia rail services in the Tuscany region
- License: CC-BY 4.0
- No registration required — direct download

**NeTEx — Trenitalia national (Italian NeTEx profile)**
- Source: Italian National Access Point (NAP), operated by CCISS
- Dataset page: https://www.cciss.it/nap/mmtis/public/catalog/Dataset/1077621
- File used: `IT-IT-TRENITALIA_L1.xml.gz` (dataset listed on the NAP as "IT-IT-TRENITALIA - Profilo NeTEx (Italia) livello 1")
- Format: `gz:xml` (gzip-compressed XML)
- Coverage: Trenitalia national rail network
- License: not explicitly stated on the NAP page

**Important note on dataset scope**

The two feeds do not cover exactly the same geographic scope. The GTFS feed covers
Trenitalia services in Tuscany only, while the NeTEx feed covers the full national
Trenitalia network. This means the comparison is not perfectly symmetric. The GTFS
side is a regional subset of what the NeTEx side covers. This will be taken into
account when interpreting the results, and lower match rates on the NeTEx side are
expected by design.

## Table of Contents

**Data Source**

**Part 1: GTFS Exploration**
- Inspect GTFS stop structure
- Prepare GTFS station table
- Inspect GTFS routes
- Prepare GTFS line table
- Inspect GTFS calendar
- Prepare GTFS calendar
- Summary of GTFS exploration

**Part 2: NeTEx Exploration**
- Inspect NeTEx file structure
- Extract all NeTEx StopPlaces
- Extract NeTEx Lines
- Inspect NeTEx calendar structure
- Inspect DayType internal structure
- Find where ValidDayBits is stored
- Inspect UicOperatingPeriod structure
- Extract all UicOperatingPeriods
- Build NeTEx active dates from UicOperatingPeriod
- Prepare NeTEx calendar summary
- Summary of NeTEx exploration

**Part 3: GTFS and NeTEx Comparison**
- 3.1 Stop-level comparison
  - Note on geographic scope
  - Restrict to geographic area
  - Coordinate proximity matching
- 3.2 Line-level comparison
  - Direct label comparison
  - Attempt to improve by keyword and transport mode
- 3.3 Calendar comparison
  - Shared date window
  - Build and compare activity patterns
  - Calendar conclusion

**Overall summary**

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import gzip
import xml.etree.ElementTree as ET
from sklearn.neighbors import BallTree
import numpy as np

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "trenitalia.gtfs"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/trenitalia.gtfs


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
3,shapes.txt,16.85
1,calendar_dates.txt,1.05
4,stop_times.txt,0.78
6,trips.txt,0.08
5,stops.txt,0.02
2,routes.txt,0.00
0,agency.txt,0.00


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (313, 5)
routes: (14, 7)
trips: (1118, 7)
calendar_dates: (37949, 3)


In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_code']


,stop_id,stop_name,stop_lat,stop_lon,stop_code
0,S01645_1,Milano P. Garibaldi,45.485032,9.187585,S01645
1,S01700_1,Milano C.Le,45.486321,9.204341,S01700
2,S01701_1,Milano Lambrate,45.484993,9.237231,S01701
3,S01820_1,Milano Rogoredo,45.433922,9.239011,S01820
4,S01825_1,Lodi,45.309150,9.497526,S01825


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color']


,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color
0,1004295345,163,NaN,Firenze - Pisa - Livorno,2,FFFFFF,0
1,1017917198,163,NaN,(Ge) La Spezia - Pisa,2,FFFFFF,0
2,1023125973,163,NaN,Siena - Chiusi,2,FFFFFF,0
3,1038909885,163,NaN,(Fi) Prato - Bologna,2,FFFFFF,0
4,1054027664,163,NaN,Firenze - Borgo Sl Via Pontassieve,2,FFFFFF,0


trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'shape_id']


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id
0,1004295345,6627_283334,6627_304556,Firenze Smn,04042,1,2594318687
1,1004295345,6627_283334,6627_304557,Firenze Smn,04026,1,2594318687
2,1004295345,6627_283334,6627_304558,Firenze Smn,18296,1,30600122
3,1004295345,6627_283334,6627_304559,Pisa C.Le,18283,0,3038614917
4,1004295345,6627_283334,6627_304560,Pisa C.Le,18285,0,3223097838


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,6627_283260,20260115,1
1,6627_283260,20260116,1
2,6627_283260,20260117,1
3,6627_283260,20260119,1
4,6627_283260,20260120,1


## Comment

The GTFS feed for Trenitalia Tuscany is small and focused. The largest file by
size is `shapes.txt` (16.85 MB), followed by `calendar_dates.txt` (1.05 MB)
and `stop_times.txt` (0.78 MB). The core scheduling files (`trips.txt`,
`stops.txt`, `routes.txt`) are very small, indicating a limited number of
routes and stops compared to the national feeds used before.

The sample stops show stations across northern and central Italy, for example Milano
Garibaldi, Milano Centrale, Milano Lambrate, Rogoredo, Lodi, confirming
that despite being labelled as a Tuscany feed, it appears to cover a wider
set of Trenitalia stops including stations in Lombardy. This is consistent
with Trenitalia running inter-regional services that cross regional boundaries.

Routes use `route_type = 2` (Rail) throughout and have no `route_short_name`
only `route_long_name` with descriptive labels like "Firenze - Pisa -
Livorno" and "Siena - Chiusi". This means the line comparison will need
to rely on long names rather than short public codes.

The calendar uses only `calendar_dates.txt` with no `calendar.txt`, consistent
with what was seen in Sweden, all active dates are defined as explicit
exception entries.

## Inspect GTFS stop structure

Before using the stop data for comparison, I check the quality and structure
of the stops table. Specifically I want to know:

- How many stops are there and are any missing names or coordinates
- Whether the feed uses `location_type` to distinguish stations from platforms
- Whether `parent_station` is used to group stops under a parent station
- Whether stop IDs are unique

In [6]:
# Inspect GTFS stop structure
stop_quality = pd.DataFrame({
    "metric": [
        "total stop rows",
        "rows with stop_name",
        "rows missing stop_name",
        "rows with stop_lat and stop_lon",
        "rows missing coordinates",
        "rows with location_type",
        "rows with parent_station",
        "unique stop_id values",
        "duplicate stop_id values"
    ],
    "value": [
        len(stops),
        stops["stop_name"].notna().sum(),
        stops["stop_name"].isna().sum(),
        stops[["stop_lat", "stop_lon"]].notna().all(axis=1).sum(),
        stops[["stop_lat", "stop_lon"]].isna().any(axis=1).sum(),
        stops["location_type"].notna().sum() if "location_type" in stops.columns else "column missing",
        stops["parent_station"].notna().sum() if "parent_station" in stops.columns else "column missing",
        stops["stop_id"].nunique(),
        stops["stop_id"].duplicated().sum()
    ]
})
display(stop_quality)

# Show location_type distribution if available
if "location_type" in stops.columns:
    print("location_type distribution:")
    display(stops["location_type"].value_counts(dropna=False).reset_index())
else:
    print("location_type column not present in this feed.")

print("Sample stops:")
display(stops.head(10))

,metric,value
0,total stop rows,313
1,rows with stop_name,313
2,rows missing stop_name,0
3,rows with stop_lat and stop_lon,313
4,rows missing coordinates,0
5,rows with location_type,column missing
6,rows with parent_station,column missing
7,unique stop_id values,313
8,duplicate stop_id values,0


location_type column not present in this feed.
Sample stops:


,stop_id,stop_name,stop_lat,stop_lon,stop_code
0,S01645_1,Milano P. Garibaldi,45.485032,9.187585,S01645
1,S01700_1,Milano C.Le,45.486321,9.204341,S01700
2,S01701_1,Milano Lambrate,45.484993,9.237231,S01701
3,S01820_1,Milano Rogoredo,45.433922,9.239011,S01820
4,S01825_1,Lodi,45.309150,9.497526,S01825
5,S01827_1,Casalpusterlengo,45.182549,9.653741,S01827
6,S01828_1,Codogno,45.155027,9.701414,S01828
7,S04220_1,Genova Sampierdarena,44.413174,8.887229,S04220
8,S04529_1,Albisola,44.334767,8.511976,S04529
9,S04530_1,Celle,44.342528,8.542473,S04530


## Comment

The GTFS stop table for Trenitalia Tuscany is small but clean:

- Only **313 stops** in total 
- All 313 stops have a name, coordinates and a unique stop ID and no missing values
- No `location_type` or `parent_station` columns which means that the feed is flat, with no
  stop hierarchy. Each row represents one station directly

The sample stops reveal an important finding about the feed's geographic scope:

- Despite being distributed through the Tuscany open data portal, the feed covers
  stations well beyond Tuscany
- Milano Garibaldi, Milano Centrale, Milano Lambrate, Lodi, Genova Sampierdarena
  are all in northern Italy
- This is expected for Trenitalia, which operates inter-regional rail services
  that cross regional boundaries

## Prepare GTFS station table

Since the Italian GTFS feed has no `location_type` or `parent_station` columns,
all 313 stops are used directly as the station-level table for comparison.

In [7]:
# Prepare GTFS stop table for Italy
# Since location_type and parent_station are not present,
# all stops are used directly as the station-level table
gtfs_stations = stops.copy()
gtfs_stations = gtfs_stations.rename(columns={
    "stop_id":   "gtfs_stop_id",
    "stop_name": "gtfs_stop_name",
    "stop_lat":  "gtfs_lat",
    "stop_lon":  "gtfs_lon"
})

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "total station rows",
        "rows with name",
        "rows missing name",
        "rows with coordinates",
        "rows missing coordinates",
        "unique station IDs"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["gtfs_stop_name"].notna().sum(),
        gtfs_stations["gtfs_stop_name"].isna().sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].notna().all(axis=1).sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].isna().any(axis=1).sum(),
        gtfs_stations["gtfs_stop_id"].nunique()
    ]
})
display(gtfs_station_quality)
display(gtfs_stations.head(10))

,metric,value
0,total station rows,313
1,rows with name,313
2,rows missing name,0
3,rows with coordinates,313
4,rows missing coordinates,0
5,unique station IDs,313


,gtfs_stop_id,gtfs_stop_name,gtfs_lat,gtfs_lon,stop_code
0,S01645_1,Milano P. Garibaldi,45.485032,9.187585,S01645
1,S01700_1,Milano C.Le,45.486321,9.204341,S01700
2,S01701_1,Milano Lambrate,45.484993,9.237231,S01701
3,S01820_1,Milano Rogoredo,45.433922,9.239011,S01820
4,S01825_1,Lodi,45.309150,9.497526,S01825
5,S01827_1,Casalpusterlengo,45.182549,9.653741,S01827
6,S01828_1,Codogno,45.155027,9.701414,S01828
7,S04220_1,Genova Sampierdarena,44.413174,8.887229,S04220
8,S04529_1,Albisola,44.334767,8.511976,S04529
9,S04530_1,Celle,44.342528,8.542473,S04530


## Comment

The station table is clean and ready for comparison:

- **313 stations**, all with names and coordinates with no missing values
- All station IDs are unique
- The sample confirms the feed covers stations across multiple Italian regions,
  not just Tuscany 

## Inspect GTFS routes

The next step is to inspect the routes table to understand how many lines the
feed contains and how they are labelled. Since the sample already showed that
`route_short_name` is missing for all routes, I also check whether
`route_long_name` can be used as a fallback label for the line comparison.

In [8]:
# Inspect GTFS route structure
route_quality = pd.DataFrame({
    "metric": [
        "total route rows",
        "routes with route_short_name",
        "routes missing route_short_name",
        "routes with route_long_name",
        "routes missing route_long_name",
        "unique route_id values",
        "duplicate route_id values"
    ],
    "value": [
        len(routes),
        routes["route_short_name"].notna().sum(),
        routes["route_short_name"].isna().sum(),
        routes["route_long_name"].notna().sum(),
        routes["route_long_name"].isna().sum(),
        routes["route_id"].nunique(),
        routes["route_id"].duplicated().sum()
    ]
})
display(route_quality)

# Show route_type distribution
print("route_type distribution:")
display(routes["route_type"].value_counts(dropna=False).reset_index())

# Show sample routes
print("Sample routes:")
display(routes[["route_id", "route_short_name", "route_long_name", "route_type"]].head(10))

,metric,value
0,total route rows,14
1,routes with route_short_name,0
2,routes missing route_short_name,14
3,routes with route_long_name,14
4,routes missing route_long_name,0
5,unique route_id values,14
6,duplicate route_id values,0


route_type distribution:


,route_type,count
0,2,14


Sample routes:


,route_id,route_short_name,route_long_name,route_type
0,1004295345,NaN,Firenze - Pisa - Livorno,2
1,1017917198,NaN,(Ge) La Spezia - Pisa,2
2,1023125973,NaN,Siena - Chiusi,2
3,1038909885,NaN,(Fi) Prato - Bologna,2
4,1054027664,NaN,Firenze - Borgo Sl Via Pontassieve,2
5,1069937666,NaN,Pisa - Livorno - Grosseto (Rm),2
6,1072814206,NaN,Firenze - Arezzo - Chiusi (Rm),2
7,1108865654,NaN,Firenze - Pistoia - Lucca - Viareggio,2
8,1145664474,NaN,Siena - Grosseto,2
9,1199544473,NaN,Pisa - Lucca - Aulla,2


## Comment

The routes table is very small and straightforward:

- Only **14 routes** in total, all with `route_type = 2` (Rail). This is a
  pure rail feed with no other transport modes
- **No `route_short_name`** for any route. All labels come from `route_long_name`
  only, using descriptive origin-destination names like "Firenze - Pisa - Livorno"
- All 14 route IDs are unique with no duplicates

The absence of `route_short_name` means the line comparison in Part 3 will have
to use `route_long_name` as the label. These long names are unlikely to match
NeTEx public codes directly, so the line match rate will probably be low similar
to what was seen for Finland.

## Prepare GTFS line table

Since all routes in the Italian feed are missing `route_short_name`, the
`route_long_name` is used as the public label for the line comparison.
This is different from all other countries analysed so far, where at least
some routes had a short public code.

In [9]:
# Prepare GTFS line table for Italy
# route_short_name is missing for all routes, so route_long_name is used as label
gtfs_lines = routes.copy()
gtfs_lines = gtfs_lines.rename(columns={
    "route_id":        "gtfs_route_id",
    "route_long_name": "gtfs_line_label",
    "route_type":      "gtfs_route_type"
})[["gtfs_route_id", "gtfs_line_label", "gtfs_route_type"]]

gtfs_line_quality = pd.DataFrame({
    "metric": [
        "total line rows",
        "rows with line label",
        "rows missing line label",
        "unique line labels",
        "duplicate line labels"
    ],
    "value": [
        len(gtfs_lines),
        gtfs_lines["gtfs_line_label"].notna().sum(),
        gtfs_lines["gtfs_line_label"].isna().sum(),
        gtfs_lines["gtfs_line_label"].nunique(),
        gtfs_lines["gtfs_line_label"].duplicated().sum()
    ]
})
display(gtfs_line_quality)
display(gtfs_lines.head(14))

,metric,value
0,total line rows,14
1,rows with line label,14
2,rows missing line label,0
3,unique line labels,14
4,duplicate line labels,0


,gtfs_route_id,gtfs_line_label,gtfs_route_type
0,1004295345,Firenze - Pisa - Livorno,2
1,1017917198,(Ge) La Spezia - Pisa,2
2,1023125973,Siena - Chiusi,2
3,1038909885,(Fi) Prato - Bologna,2
4,1054027664,Firenze - Borgo Sl Via Pontassieve,2
5,1069937666,Pisa - Livorno - Grosseto (Rm),2
6,1072814206,Firenze - Arezzo - Chiusi (Rm),2
7,1108865654,Firenze - Pistoia - Lucca - Viareggio,2
8,1145664474,Siena - Grosseto,2
9,1199544473,Pisa - Lucca - Aulla,2


## Comment

The GTFS line table contains only **14 lines**, all with a unique label and no
missing values. All lines are rail (`route_type = 2`).


In [10]:
# Check route_type distribution in detail
print("route_type distribution:")
display(
    routes["route_type"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"route_type": "route_type", "count": "n_routes"})
)

# Map route type to name
route_type_names = {
    0:   "Tram",
    1:   "Metro / Subway",
    2:   "Rail",
    3:   "Bus",
    4:   "Ferry",
    100: "Railway Service",
    101: "High Speed Rail",
    102: "Long Distance Rail",
    106: "Regional Rail",
    109: "Suburban Rail"
}

routes["route_type_name"] = routes["route_type"].map(route_type_names)
display(routes[["route_id", "route_long_name", "route_type", "route_type_name"]])

route_type distribution:


,route_type,n_routes
0,2,14


,route_id,route_long_name,route_type,route_type_name
0,1004295345,Firenze - Pisa - Livorno,2,Rail
1,1017917198,(Ge) La Spezia - Pisa,2,Rail
2,1023125973,Siena - Chiusi,2,Rail
3,1038909885,(Fi) Prato - Bologna,2,Rail
4,1054027664,Firenze - Borgo Sl Via Pontassieve,2,Rail
5,1069937666,Pisa - Livorno - Grosseto (Rm),2,Rail
6,1072814206,Firenze - Arezzo - Chiusi (Rm),2,Rail
7,1108865654,Firenze - Pistoia - Lucca - Viareggio,2,Rail
8,1145664474,Siena - Grosseto,2,Rail
9,1199544473,Pisa - Lucca - Aulla,2,Rail


## Comment

All 14 routes use `route_type = 2`, which corresponds to standard rail in the
basic GTFS specification. 

## Inspect GTFS calendar

Since the feed contains no `calendar.txt`, all service validity is defined
through `calendar_dates.txt` only. I inspect the date range, the number of
service IDs, and the exception types used.

In [11]:
# Convert date column to datetime
calendar_dates["date_dt"] = pd.to_datetime(calendar_dates["date"], format="%Y%m%d", errors="coerce")

# Inspect calendar structure
calendar_quality = pd.DataFrame({
    "metric": [
        "total calendar_dates rows",
        "unique service_id values",
        "exception type 1 rows (added dates)",
        "exception type 2 rows (removed dates)",
        "earliest date",
        "latest date",
        "number of days in feed window"
    ],
    "value": [
        len(calendar_dates),
        calendar_dates["service_id"].nunique(),
        (calendar_dates["exception_type"] == 1).sum(),
        (calendar_dates["exception_type"] == 2).sum(),
        calendar_dates["date_dt"].min(),
        calendar_dates["date_dt"].max(),
        (calendar_dates["date_dt"].max() - calendar_dates["date_dt"].min()).days + 1
    ]
})
display(calendar_quality)

print("Sample calendar_dates:")
display(calendar_dates.head(10))

,metric,value
0,total calendar_dates rows,37949
1,unique service_id values,231
2,exception type 1 rows (added dates),37949
3,exception type 2 rows (removed dates),0
4,earliest date,2026-01-15 00:00:00
5,latest date,2026-12-12 00:00:00
6,number of days in feed window,332


Sample calendar_dates:


,service_id,date,exception_type,date_dt
0,6627_283260,20260115,1,2026-01-15
1,6627_283260,20260116,1,2026-01-16
2,6627_283260,20260117,1,2026-01-17
3,6627_283260,20260119,1,2026-01-19
4,6627_283260,20260120,1,2026-01-20
5,6627_283260,20260121,1,2026-01-21
6,6627_283260,20260122,1,2026-01-22
7,6627_283260,20260123,1,2026-01-23
8,6627_283260,20260124,1,2026-01-24
9,6627_283260,20260126,1,2026-01-26


## Comment

The calendar structure for the Italian GTFS feed is simple and clean:

- **37,949 rows** in `calendar_dates.txt` covering **231 unique service IDs**
- **All rows are exception type 1** (added dates) so there are no removed dates
  at all, meaning every active date is explicitly listed as an addition
- The feed covers a window of **332 days**, from 2026-01-15 to 2026-12-12 —
  almost a full year
- The sample shows that service IDs follow a structured format like
  `6627_283260`, with consecutive daily entries suggesting regular weekday
  services

The absence of exception type 2 rows and the use of `calendar_dates.txt` only
(no `calendar.txt`) is consistent with what was seen in Sweden and Finland.
All active dates are defined explicitly rather than through weekly patterns.
This makes the active date reconstruction straightforward: all type 1 rows
are active dates with no removals to process.

## Prepare GTFS calendar

Since the Italian feed uses only `calendar_dates.txt` with no `calendar.txt`
and no exception type 2 rows, building the active dates is straightforward because
every row is an active date. No weekly pattern expansion or removal of
exception dates is needed.

In [12]:
# Build GTFS active dates dictionary from calendar_dates.txt
# All rows are exception type 1 (added dates) — no removals to process
gtfs_active_dates = (
    calendar_dates[calendar_dates["exception_type"] == 1]
    .groupby("service_id")["date_dt"]
    .apply(set)
    .to_dict()
)

print(f"GTFS active dates built for {len(gtfs_active_dates)} service IDs.")

# Summary
gtfs_calendar_summary = (
    calendar_dates[calendar_dates["exception_type"] == 1]
    .groupby("service_id", as_index=False)
    .agg(
        n_active_dates=("date_dt", "nunique"),
        first_active_date=("date_dt", "min"),
        last_active_date=("date_dt", "max")
    )
)

display(gtfs_calendar_summary.head(10))

GTFS active dates built for 231 service IDs.


,service_id,n_active_dates,first_active_date,last_active_date
0,6627_283260,279,2026-01-15,2026-12-12
1,6627_283261,256,2026-01-15,2026-12-12
2,6627_283262,53,2026-01-18,2026-12-08
3,6627_283263,300,2026-01-17,2026-12-12
4,6627_283264,233,2026-01-15,2026-12-11
5,6627_283265,278,2026-01-15,2026-12-11
6,6627_283266,332,2026-01-15,2026-12-12
7,6627_283267,16,2026-05-17,2026-08-16
8,6627_283268,213,2026-01-15,2026-12-11
9,6627_283269,28,2026-04-05,2026-09-06


## Comment

The active dates dictionary was built successfully for **231 service IDs**:

- Most service IDs cover a large part of the year — service `6627_283266`
  runs on all **332 days** of the feed window, while others are more limited
    
- Some services are very short — `6627_283267` runs on only **16 days**
  between May and August, likely a seasonal summer service
    
- All service IDs follow the same `6627_XXXXXX` format, suggesting they
  all belong to the same operator block
    
- First and last active dates are consistent with the feed window of
  2026-01-15 to 2026-12-12 seen earlier



## Summary of GTFS exploration 

The Italian GTFS feed is the smallest dataset analysed so far. Key findings:

**Stops:**
- 313 stops, all with names and coordinates 
- No `location_type` or `parent_station`, flat structure, all stops are
  station-level records
- Coverage extends beyond Tuscany to northern Italy (Milano, Genova) due to
  Trenitalia's inter-regional services

**Lines:**
- Only 14 routes, all rail (`route_type = 2`)
- No `route_short_name`, `route_long_name` used as label instead
- Labels are descriptive origin-destination strings like "Firenze - Pisa - Livorno"


**Calendar:**
- 231 service IDs covering a window of 332 days (2026-01-15 to 2026-12-12)
- Only `calendar_dates.txt` used, no `calendar.txt` and no exception type 2 rows
- All active dates are explicitly listed, making reconstruction straightforward


# Part 2: NeTEx exploration

In [13]:
netex_path = data_dir / "IT-IT-TRENITALIA_L1.xml.gz"

print("NeTEx exists:", netex_path.exists())
print(f"File size: {netex_path.stat().st_size / (1024*1024):.1f} MB")

# Peek at the first 5000 bytes without fully decompressing
with gzip.open(netex_path, "rb") as f:
    head = f.read(5000)

print(head.decode("utf-8", errors="replace"))

NeTEx exists: True
File size: 15.5 MB
<?xml version="1.0" encoding="UTF-8"?>
				<!-- NeTEX italian Profile -->
				<PublicationDelivery xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:siri="http://www.siri.org.uk/siri" version="1.10" xmlns="http://www.netex.org.uk/netex" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.netex.org.uk/netex ../../../netex-italian-profile-main/xsd/NeTEx_publication_EPIP.xsd">
					<PublicationTimestamp>2026-05-14T14:41:36Z</PublicationTimestamp>
					<ParticipantRef>IT-RAP</ParticipantRef>
					<Description>IT TRENITALIA NeTEx Italian Profile</Description>
					<dataObjects>
						<CompositeFrame id="epd:it:CompositeFrame_EU_PI_STOP_OFFER:ita" version="1">
							<ValidBetween>
								<FromDate>2025-12-14T00:00:00.000+02:00</FromDate>
								<ToDate>2026-06-13T23:59:59.999+02:00</ToDate>
							</ValidBetween>
							<TypeOfFrameRef ref="epip:EU_PI_LINE_OFFER" versionRef="1"/>
							<!--- ======= CODESPACEs======== 

## Output

A quick peek at the first bytes of the gzip-compressed NeTEx file reveals the
following:

- The file follows the **Italian NeTEx profile** (`NeTEx italian Profile`)
- It is a single gzip-compressed XML file 
- **Operator**: Trenitalia only
- **Validity window**: 2025-12-14 to 2026-06-13 — about 6 months, shorter
  than the GTFS feed window of 332 days
- **StopPlace IDs** follow the format `IT::StopPlace:otherTRENITALIA:...`
  — different from the GTFS stop IDs, so coordinate proximity matching will
  be needed
- **Coordinates** are nested under `<Centroid><Location><Latitude>` rather
  than directly under the StopPlace 
- Each StopPlace contains one `Quay` — same flat structure as Finland
- **`TransportMode` is "other"** for all sampled stops — no meaningful mode
  classification available in the stop data

In [14]:
def local_name(tag):
    # XML tags in NeTEx include a namespace prefix like {http://www.netex.org.uk/netex}StopPlace
    # This function strips the namespace and returns only the tag name (e.g. "StopPlace")
    return tag.split("}")[-1] if "}" in tag else tag

In [15]:
# Count key NeTEx elements in the Italian file

target_tags = [
    "StopPlace",
    "Quay",
    "Line",
    "Route",
    "ServiceJourney",
    "DayType",
    "DayTypeAssignment",
    "OperatingPeriod",
    "ValidDayBits"
]

counts = {tag: 0 for tag in target_tags}
events_scanned = 0

with gzip.open(netex_path, "rb") as f:
    for event, elem in ET.iterparse(f, events=("end",)):
        tag = local_name(elem.tag)
        events_scanned += 1
        if events_scanned % 100000 == 0:
            print(f"  {events_scanned:,} events scanned...", end="\r")
        if tag in counts:
            counts[tag] += 1
        elem.clear()

print(f"\nDone — {events_scanned:,} total events scanned.")

netex_structure_check = pd.DataFrame({
    "tag":   list(counts.keys()),
    "count": list(counts.values())
})
display(netex_structure_check)

  7,100,000 events scanned...
Done — 7,103,809 total events scanned.


,tag,count
0,StopPlace,1910
1,Quay,1910
2,Line,15
3,Route,15
4,ServiceJourney,33054
5,DayType,33054
6,DayTypeAssignment,33054
7,OperatingPeriod,0
8,ValidDayBits,33054


## Comment

**Stops:**
- **1,910 StopPlaces** and exactly **1,910 Quays** — one Quay per StopPlace,
  confirming the flat structure seen in the peek. This is much larger than the
  313 GTFS stops, meaning NeTEx covers the full national Trenitalia network
  while GTFS only covers Tuscany

**Lines:**
- **15 Lines** and **15 Routes** — very close to the 14 GTFS routes, which is
  encouraging for the line comparison

**Calendar:**
- **ValidDayBits** is used — 33,054 occurrences — this is the same approach
  as Luxembourg and France, where calendar is encoded as a binary bit string
- **No OperatingPeriod** at all — so Italy does not use date ranges 
- **33,054 DayTypes, DayTypeAssignments and ValidDayBits** — all the same count,
  meaning every ServiceJourney has exactly one DayType with one assignment and
  one ValidDayBits string

This means the calendar comparison for Italy will follow the **Luxembourg/France
approach** which implies reconstructing active dates from ValidDayBits bit strings.

## Extract all NeTEx StopPlaces

All StopPlace elements are extracted from the Italian NeTEx file. The extraction
function is adapted from the Norway and Sweden version with two changes:

- The file is gzip-compressed directly, not wrapped in a ZIP so it is opened
  with `gzip.open` instead of `zipfile.ZipFile`
- Coordinates are nested under `<Centroid><Location>` rather than directly
  under the StopPlace. The same `Latitude` and `Longitude` tag names are used,
  just deeper in the structure, which the depth-tracking pattern handles correctly

In [16]:
def extract_all_stopplaces(zip_path_or_gzip, xml_file=None):
    rows = []
    current = None
    depth = 0

    # Handle both gzip and zip
    if xml_file is None:
        f_context = gzip.open(zip_path_or_gzip, "rb")
    else:
        import zipfile
        f_context = zipfile.ZipFile(zip_path_or_gzip).open(xml_file)

    with f_context as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)

            if event == "start" and tag == "StopPlace":
                depth += 1
                if depth == 1:
                    current = {
                        "netex_stopplace_id":   elem.attrib.get("id"),
                        "netex_stopplace_name": None,
                        "netex_lat":            None,
                        "netex_lon":            None
                    }

            elif event == "end" and tag == "StopPlace":
                if depth == 1 and current is not None:
                    rows.append(current)
                    if len(rows) % 200 == 0:
                        print(f"  {len(rows)} StopPlaces extracted...", end="\r")
                    current = None
                depth -= 1
                elem.clear()

            elif event == "end" and current is not None and depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "Name" and current["netex_stopplace_name"] is None:
                    current["netex_stopplace_name"] = text
                elif tag == "Latitude" and current["netex_lat"] is None:
                    current["netex_lat"] = text
                elif tag == "Longitude" and current["netex_lon"] is None:
                    current["netex_lon"] = text
                elem.clear()

            elif event == "end":
                elem.clear()

    print(f"\nDone — {len(rows)} StopPlaces extracted.")
    return pd.DataFrame(rows)

netex_stopplaces = extract_all_stopplaces(netex_path)

netex_stopplaces["netex_lat"] = pd.to_numeric(netex_stopplaces["netex_lat"], errors="coerce")
netex_stopplaces["netex_lon"] = pd.to_numeric(netex_stopplaces["netex_lon"], errors="coerce")

display(netex_stopplaces.head(10))

  1800 StopPlaces extracted...
Done — 1910 StopPlaces extracted.


,netex_stopplace_id,netex_stopplace_name,netex_lat,netex_lon
0,IT::StopPlace:otherTRENITALIA:830006900,FIRENZE CAMPO MARTE,43.776837,11.277023
1,IT::StopPlace:otherTRENITALIA:830006421,FIRENZE S.MARIA NOVELLA,43.776874,11.247383
2,IT::StopPlace:otherTRENITALIA:830006420,FIRENZE RIFREDI,43.800855,11.235735
3,IT::StopPlace:otherTRENITALIA:830005009,FIDENZA,44.867733,10.063396
4,IT::StopPlace:otherTRENITALIA:830005000,PIACENZA,45.052241,9.706339
5,IT::StopPlace:otherTRENITALIA:830001825,LODI,45.308966,9.497588
6,IT::StopPlace:otherTRENITALIA:830001820,MILANO ROGOREDO,45.433862,9.239110
7,IT::StopPlace:otherTRENITALIA:830001700,MILANO CENTRALE,45.486341,9.204544
8,IT::StopPlace:otherTRENITALIA:830006414,PISTOIA,43.926794,10.914634
9,IT::StopPlace:otherTRENITALIA:830006411,MONTECATINI TERME,43.879323,10.780595


## Comment

Names and coordinates are available for all
sampled StopPlaces. A few observations:

- **1,910 StopPlaces** extracted, confirming the structure check count
- The sample shows stations across Italy like for example Firenze, Piacenza, Lodi, Milano,
  Pistoia, Montecatini Terme, confirming national coverage
- Coordinates look correct with latitudes between 43° and 45°N, longitudes
  between 9° and 11°E, all within Italian territory

In [17]:
# NeTEx StopPlace data quality check
netex_stopplace_quality = pd.DataFrame({
    "metric": [
        "total StopPlace rows",
        "rows with name",
        "rows missing name",
        "rows with coordinates",
        "rows missing coordinates",
        "unique StopPlace IDs",
        "duplicate StopPlace IDs"
    ],
    "value": [
        len(netex_stopplaces),
        netex_stopplaces["netex_stopplace_name"].notna().sum(),
        netex_stopplaces["netex_stopplace_name"].isna().sum(),
        netex_stopplaces[["netex_lat", "netex_lon"]].notna().all(axis=1).sum(),
        netex_stopplaces[["netex_lat", "netex_lon"]].isna().any(axis=1).sum(),
        netex_stopplaces["netex_stopplace_id"].nunique(),
        netex_stopplaces["netex_stopplace_id"].duplicated().sum()
    ]
})
display(netex_stopplace_quality)

,metric,value
0,total StopPlace rows,1910
1,rows with name,1910
2,rows missing name,0
3,rows with coordinates,1910
4,rows missing coordinates,0
5,unique StopPlace IDs,1910
6,duplicate StopPlace IDs,0


## Comment

The NeTEx StopPlace table is complete and clean:

- All **1,910 StopPlaces** have a name, coordinates and a unique ID
- No missing values and no duplicate IDs


## Extract NeTEx Lines

Since the Italian NeTEx is a single gzip file rather than multiple line-specific
XML files, all Line elements are extracted in one pass. I first extract a sample
to inspect the structure and confirm which fields are available before running
the full extraction.

In [18]:
# Extract a sample of NeTEx Line elements from the Italian gzip file
def extract_line_sample_gzip(gzip_path, n_sample=15):
    rows = []
    current = None
    depth = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)

            if event == "start" and tag == "Line":
                depth += 1
                if depth == 1:
                    current = {
                        "netex_line_id":        elem.attrib.get("id"),
                        "netex_line_name":      None,
                        "netex_short_name":     None,
                        "netex_public_code":    None,
                        "netex_transport_mode": None
                    }

            elif event == "end" and tag == "Line":
                if depth == 1 and current is not None:
                    rows.append(current)
                    current = None
                depth -= 1
                elem.clear()
                if len(rows) >= n_sample:
                    break

            elif event == "end" and current is not None and depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "Name" and current["netex_line_name"] is None:
                    current["netex_line_name"] = text
                elif tag == "ShortName" and current["netex_short_name"] is None:
                    current["netex_short_name"] = text
                elif tag == "PublicCode" and current["netex_public_code"] is None:
                    current["netex_public_code"] = text
                elif tag == "TransportMode" and current["netex_transport_mode"] is None:
                    current["netex_transport_mode"] = text
                elem.clear()

            elif event == "end":
                elem.clear()

    return pd.DataFrame(rows)

netex_line_sample = extract_line_sample_gzip(netex_path, n_sample=15)
display(netex_line_sample)

,netex_line_id,netex_line_name,netex_short_name,netex_public_code,netex_transport_mode
0,IT::Line:busTRENITALIA:70083,Autobus,BUS,BUS,bus
1,IT::Line:railTRENITALIA:10083,Regionale,REG,REG,rail
2,IT::Line:railTRENITALIA:620083,Regionale Veloce,RV,RV,rail
3,IT::Line:railTRENITALIA:80083,Metropolitano,MET,MET,rail
4,IT::Line:railTRENITALIA:60083,Eurocity,EC,EC,rail
5,IT::Line:railTRENITALIA:120083,Euronight,EN,EN,rail
6,IT::Line:railTRENITALIA:720083,sfm,SFM,SFM,rail
7,IT::Line:railTRENITALIA:50083,Intercity,IC,IC,rail
8,IT::Line:railTRENITALIA:680083,Frecciarossa,FR,FR,rail
9,IT::Line:railTRENITALIA:90083,FrecciaLink,FL,FL,rail


## Comment

The NeTEx line sample reveals an important structural difference from GTFS:

- NeTEx models lines by **service type** rather than by route. There are 15
  lines total representing Trenitalia's service categories: Regionale (REG),
  Regionale Veloce (RV), Intercity (IC), Frecciarossa (FR), Frecciargento (FA),
  Frecciabianca (FB), Eurocity (EC), Euronight (EN) and others
- All lines have a `PublicCode` and `ShortName` unlike GTFS where no
  `route_short_name` was present
- One line is bus (`busTRENITALIA`), Trenitalia also operates replacement
  bus services, which is why `route_type = 2` in GTFS did not capture this
- The ID format `IT::Line:railTRENITALIA:...` is consistent with the StopPlace
  ID format seen earlier

This reveals a fundamental difference in how GTFS and NeTEx model Trenitalia's
network. GTFS uses **geographic route lines** (origin-destination pairs like
"Firenze - Pisa - Livorno") while NeTEx uses **service type lines** (categories
like "Regionale" or "Frecciarossa"). These two approaches are not directly
comparable by label, which will significantly affect the line comparison in Part 3.

## Extract all 15 NeTEx lines fully and prepare the line table for comparison

In [19]:
# Extract all NeTEx Line elements from the Italian gzip file
def extract_all_lines_gzip(gzip_path):
    rows = []
    current = None
    depth = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)

            if event == "start" and tag == "Line":
                depth += 1
                if depth == 1:
                    current = {
                        "netex_line_id":        elem.attrib.get("id"),
                        "netex_line_name":      None,
                        "netex_short_name":     None,
                        "netex_public_code":    None,
                        "netex_transport_mode": None
                    }

            elif event == "end" and tag == "Line":
                if depth == 1 and current is not None:
                    rows.append(current)
                    current = None
                depth -= 1
                elem.clear()

            elif event == "end" and current is not None and depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "Name" and current["netex_line_name"] is None:
                    current["netex_line_name"] = text
                elif tag == "ShortName" and current["netex_short_name"] is None:
                    current["netex_short_name"] = text
                elif tag == "PublicCode" and current["netex_public_code"] is None:
                    current["netex_public_code"] = text
                elif tag == "TransportMode" and current["netex_transport_mode"] is None:
                    current["netex_transport_mode"] = text
                elem.clear()

            elif event == "end":
                elem.clear()

    print(f"Done — {len(rows)} Lines extracted.")
    return pd.DataFrame(rows)

netex_lines = extract_all_lines_gzip(netex_path)
display(netex_lines)

Done — 15 Lines extracted.


,netex_line_id,netex_line_name,netex_short_name,netex_public_code,netex_transport_mode
0,IT::Line:busTRENITALIA:70083,Autobus,BUS,BUS,bus
1,IT::Line:railTRENITALIA:10083,Regionale,REG,REG,rail
2,IT::Line:railTRENITALIA:620083,Regionale Veloce,RV,RV,rail
3,IT::Line:railTRENITALIA:80083,Metropolitano,MET,MET,rail
4,IT::Line:railTRENITALIA:60083,Eurocity,EC,EC,rail
5,IT::Line:railTRENITALIA:120083,Euronight,EN,EN,rail
6,IT::Line:railTRENITALIA:720083,sfm,SFM,SFM,rail
7,IT::Line:railTRENITALIA:50083,Intercity,IC,IC,rail
8,IT::Line:railTRENITALIA:680083,Frecciarossa,FR,FR,rail
9,IT::Line:railTRENITALIA:90083,FrecciaLink,FL,FL,rail


## Comment

All 15 NeTEx Lines extracted successfully. The full line table confirms what
the sample already showed:

- **14 rail lines** and **1 bus line** (Autobus) matching the structure check count
- All lines have a `PublicCode` and `ShortName` which are identical, for example
  "REG" for Regionale and "FR" for Frecciarossa
- The lines represent Trenitalia's **service categories**, not geographic routes which
  is fundamentally different from the 14 GTFS routes which are origin-destination
  pairs specific to Tuscany
- Notable service types include high-speed trains (Frecciarossa FR, Frecciargento FA,
  Frecciabianca FB), intercity services (IC, ICN, EC, EN) and regional services
  (REG, RV)

The line table is ready but the comparison with GTFS will be challenging because NeTEx
uses short service type codes (REG, FR, IC) while GTFS uses long geographic names
(Firenze - Pisa - Livorno). These two labelling systems are not directly comparable.

## Inspect NeTEx calendar structure

Since the structure check showed that Italy uses `ValidDayBits`, the same
approach as Luxembourg and France, I now inspect a sample of `DayType` and
`DayTypeAssignment` elements to understand how the calendar is structured
and confirm the ValidDayBits format before extracting the full calendar.

In [20]:
# Inspect a sample of DayType and DayTypeAssignment elements
def extract_calendar_sample_gzip(gzip_path, n_sample=5):
    daytype_rows = []
    assignment_rows = []
    current_dt = None
    dt_depth = 0
    current_as = None
    as_depth = 0
    events_scanned = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            events_scanned += 1

            if events_scanned % 100000 == 0:
                print(f"  {events_scanned:,} events scanned", end="\r")

            # DayType
            if event == "start" and tag == "DayType":
                dt_depth += 1
                if dt_depth == 1:
                    current_dt = {
                        "daytype_id":    elem.attrib.get("id"),
                        "valid_day_bits": None
                    }
            elif event == "end" and tag == "DayType":
                if dt_depth == 1 and current_dt is not None:
                    if len(daytype_rows) < n_sample:
                        daytype_rows.append(current_dt)
                    current_dt = None
                dt_depth -= 1
                elem.clear()
            elif event == "end" and current_dt is not None and dt_depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "ValidDayBits" and current_dt["valid_day_bits"] is None:
                    current_dt["valid_day_bits"] = text
                elem.clear()

            # DayTypeAssignment
            elif event == "start" and tag == "DayTypeAssignment":
                as_depth += 1
                if as_depth == 1:
                    current_as = {
                        "assignment_id":  elem.attrib.get("id"),
                        "daytype_ref":    None,
                        "from_date":      None,
                        "to_date":        None
                    }
            elif event == "end" and tag == "DayTypeAssignment":
                if as_depth == 1 and current_as is not None:
                    if len(assignment_rows) < n_sample:
                        assignment_rows.append(current_as)
                    current_as = None
                as_depth -= 1
                elem.clear()
            elif event == "end" and current_as is not None and as_depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "DayTypeRef":
                    current_as["daytype_ref"] = elem.attrib.get("ref")
                elif tag == "FromDate" and current_as["from_date"] is None:
                    current_as["from_date"] = text
                elif tag == "ToDate" and current_as["to_date"] is None:
                    current_as["to_date"] = text
                elem.clear()

            elif event == "end":
                elem.clear()

            if len(daytype_rows) >= n_sample and len(assignment_rows) >= n_sample:
                break

    print(f"\nDone.")
    return pd.DataFrame(daytype_rows), pd.DataFrame(assignment_rows)

netex_daytype_sample, netex_assignment_sample = extract_calendar_sample_gzip(
    netex_path, n_sample=5
)
print("DayType sample:")
display(netex_daytype_sample)
print("DayTypeAssignment sample:")
display(netex_assignment_sample)

  800,000 events scanned
Done.
DayType sample:


,daytype_id,valid_day_bits
0,IT::DayType:TRENITALIA:10-5A01-0083-6,None
1,IT::DayType:TRENITALIA:10-5A02-0083-6,None
2,IT::DayType:TRENITALIA:10-5A03-0083-6,None
3,IT::DayType:TRENITALIA:10-5A04-0083-6,None
4,IT::DayType:TRENITALIA:10201-59EF-0083-1,None


DayTypeAssignment sample:


,assignment_id,daytype_ref,from_date,to_date
0,IT::DayTypeAssignment:TRENITALIA:2022-01-01_10...,IT::DayType:TRENITALIA:10-5A01-0083-6,None,None
1,IT::DayTypeAssignment:TRENITALIA:2022-01-01_10...,IT::DayType:TRENITALIA:10-5A02-0083-6,None,None
2,IT::DayTypeAssignment:TRENITALIA:2022-01-01_10...,IT::DayType:TRENITALIA:10-5A03-0083-6,None,None
3,IT::DayTypeAssignment:TRENITALIA:2022-01-01_10...,IT::DayType:TRENITALIA:10-5A04-0083-6,None,None
4,IT::DayTypeAssignment:TRENITALIA:2022-01-01_10...,IT::DayType:TRENITALIA:10201-59EF-0083-1,None,None


## Comment

The sample shows an unexpected result: `ValidDayBits` is `None` for all sampled
DayTypes, and `FromDate`/`ToDate` are also `None` for all DayTypeAssignments.
This means the calendar data is not being extracted correctly with the current
approach.

The structure check confirmed 33,054 `ValidDayBits` elements exist in the file,
so the data is there but the extraction is not finding it inside the `DayType`
elements. This suggests that `ValidDayBits` may be nested differently in the
Italian NeTEx profile compared to Luxembourg and France, or it may be stored
as an attribute rather than as text content.

The next step is to inspect the raw XML structure of a DayType element to find
exactly where `ValidDayBits` is stored.

## Inspect DayType internal structure

Since `ValidDayBits` was not found inside the sampled DayType elements, I now
inspect the raw child tag structure of a DayType to find where the calendar
bit string is actually stored in the Italian NeTEx profile.

In [21]:
# Inspect the internal structure of the first few DayType elements
def inspect_daytype_structure_gzip(gzip_path, n_sample=3):
    rows = []
    found = 0
    current_id = None
    depth = 0
    events_scanned = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            events_scanned += 1

            if events_scanned % 100000 == 0:
                print(f"  {events_scanned:,} events scanned — {found}/{n_sample} DayTypes found", end="\r")

            if event == "start" and tag == "DayType":
                depth += 1
                if depth == 1:
                    current_id = elem.attrib.get("id")
                    found += 1

            elif event == "end" and tag == "DayType":
                if depth == 1:
                    current_id = None
                depth -= 1
                elem.clear()
                if found >= n_sample:
                    break

            elif event == "end" and current_id is not None and depth > 0:
                text = elem.text.strip() if elem.text and elem.text.strip() else None
                rows.append({
                    "daytype_id":       current_id,
                    "child_tag":        tag,
                    "child_text":       text,
                    "child_attributes": dict(elem.attrib)
                })
                elem.clear()

            elif event == "end":
                elem.clear()

    print(f"\nDone — {found} DayTypes inspected.")
    return pd.DataFrame(rows)

daytype_structure = inspect_daytype_structure_gzip(netex_path, n_sample=3)
display(daytype_structure)


Done — 3 DayTypes inspected.


,daytype_id,child_tag,child_text,child_attributes
0,IT::DayType:TRENITALIA:10-5A01-0083-6,Name,10-5A01-0083-6,{}
1,IT::DayType:TRENITALIA:10-5A01-0083-6,Description,10-5A01-0083-6,{}
2,IT::DayType:TRENITALIA:10-5A01-0083-6,DaysOfWeek,Thursday Friday Saturday,{}
3,IT::DayType:TRENITALIA:10-5A01-0083-6,HolidayTypes,None,{}
4,IT::DayType:TRENITALIA:10-5A01-0083-6,PropertyOfDay,None,{}
5,IT::DayType:TRENITALIA:10-5A01-0083-6,properties,None,{}
6,IT::DayType:TRENITALIA:10-5A02-0083-6,Name,10-5A02-0083-6,{}
7,IT::DayType:TRENITALIA:10-5A02-0083-6,Description,10-5A02-0083-6,{}
8,IT::DayType:TRENITALIA:10-5A02-0083-6,DaysOfWeek,Sunday Monday Tuesday Wednesday Thursday Frida...,{}
9,IT::DayType:TRENITALIA:10-5A02-0083-6,HolidayTypes,None,{}


## Comment

The DayType structure inspection reveals that the Italian NeTEx profile uses a
completely different calendar approach than expected:

- There are **no `ValidDayBits`** inside DayType elements despite the structure
  check counting 33,054 of them so they must be stored elsewhere in the file
- Instead, DayTypes store a **`DaysOfWeek`** field listing the days a service
  runs, for example "Thursday Friday Saturday" or "Sunday Monday Tuesday
  Wednesday Thursday Friday Saturday"
- This is a **weekday pattern approach** similar in concept to `calendar.txt`
  in GTFS, where each service is defined by which days of the week it runs
- The `DayTypeAssignment` must then link each DayType to a date range to define
  the full validity period

This means the Italian NeTEx calendar is structured as:
**DayType (weekday pattern) + DayTypeAssignment (date range) = active dates**

The `ValidDayBits` found in the structure check must belong to a different element
likely `ServiceJourney` or `UicOperatingPeriod`. The next step is to find where
`ValidDayBits` actually appears in the file.

## Find where ValidDayBits is stored

Since `ValidDayBits` does not appear inside `DayType` elements, I search for
its parent element to understand how it fits into the Italian NeTEx calendar
structure.

In [22]:
# Find the parent element of ValidDayBits in the Italian NeTEx file
def find_validaybit_parent(gzip_path, n_sample=5):
    rows = []
    parent_stack = []
    events_scanned = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            events_scanned += 1

            if events_scanned % 100000 == 0:
                print(f"  {events_scanned:,} events scanned — {len(rows)} found", end="\r")

            if event == "start":
                parent_stack.append(tag)

            elif event == "end":
                if tag == "ValidDayBits" and len(rows) < n_sample:
                    text = elem.text.strip() if elem.text else None
                    rows.append({
                        "parent_tag":    parent_stack[-2] if len(parent_stack) >= 2 else None,
                        "valid_day_bits": text[:50] if text else None
                    })
                parent_stack.pop()
                elem.clear()

            if len(rows) >= n_sample:
                break

    print(f"\nDone.")
    return pd.DataFrame(rows)

validaybit_parents = find_validaybit_parent(netex_path, n_sample=5)
display(validaybit_parents)

  500,000 events scanned — 0 found
Done.


,parent_tag,valid_day_bits
0,UicOperatingPeriod,1000000000000000000000000000000000000000010000...
1,UicOperatingPeriod,1111111111111111111111111111100000000000011111...
2,UicOperatingPeriod,11
3,UicOperatingPeriod,111111111111
4,UicOperatingPeriod,11100000000000011000001100000110100011000001


## Comment

The `ValidDayBits` elements are stored inside `UicOperatingPeriod` elements,
not inside `DayType` as seen in Luxembourg and France. 

The Italian calendar therefore works as follows:

- `DayType` stores the **weekday pattern** (e.g. "Thursday Friday Saturday")
- `UicOperatingPeriod` stores a **`ValidDayBits` bit string** encoding the
  specific days the service runs within a date range
- `DayTypeAssignment` links the two together

The bit strings visible in the sample look correct, sequences of 1s and 0s
representing active and inactive days. However row 2 has only "11" and row 3
has "111111111111", which are very short. These may represent services with
a very limited validity window.

This structure is more complex than Luxembourg and France where `ValidDayBits`
was directly inside `DayType`. The calendar extraction will need to handle the
`UicOperatingPeriod` level and link it back to the correct `DayType` via
`DayTypeAssignment`.

## Inspect UicOperatingPeriod structure

Now that `ValidDayBits` is confirmed to be inside `UicOperatingPeriod`, I inspect
a sample of these elements to understand what other fields they contain. In
particular the `FromDate` and `ToDate` needed to anchor the bit string to
actual calendar dates.

In [23]:
def inspect_uic_operating_period_gzip(gzip_path, n_sample=5):
    rows = []
    current_id = None
    depth = 0
    events_scanned = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            events_scanned += 1

            if events_scanned % 100000 == 0:
                print(f"  {events_scanned:,} events scanned — {len(rows)}/{n_sample} found", end="\r")

            if event == "start" and tag == "UicOperatingPeriod":
                depth += 1
                if depth == 1:
                    current_id = elem.attrib.get("id")

            elif event == "end" and tag == "UicOperatingPeriod":
                if depth == 1:
                    current_id = None
                depth -= 1
                elem.clear()
                if len(rows) >= n_sample:
                    break

            elif event == "end" and current_id is not None and depth > 0:
                text = elem.text.strip() if elem.text and elem.text.strip() else None
                rows.append({
                    "uic_period_id": current_id,
                    "child_tag":     tag,
                    "child_text":    text[:60] if text else None,
                    "attributes":    dict(elem.attrib)
                })
                elem.clear()

            elif event == "end":
                elem.clear()

    print(f"\nDone — {len(rows)} child elements inspected.")
    return pd.DataFrame(rows)

uic_period_structure = inspect_uic_operating_period_gzip(netex_path, n_sample=5)
display(uic_period_structure)

  500,000 events scanned — 0/5 found
Done — 6 child elements inspected.


,uic_period_id,child_tag,child_text,attributes
0,IT::UicOperatingPeriod:TRENITALIA:10-5A01-0083-6,FromDate,2026-04-03T00:00:00,{}
1,IT::UicOperatingPeriod:TRENITALIA:10-5A01-0083-6,ToDate,2026-05-23T23:59:59,{}
2,IT::UicOperatingPeriod:TRENITALIA:10-5A01-0083-6,ValidDayBits,1000000000000000000000000000000000000000010000...,{}
3,IT::UicOperatingPeriod:TRENITALIA:10-5A02-0083-6,FromDate,2025-12-14T00:00:00,{}
4,IT::UicOperatingPeriod:TRENITALIA:10-5A02-0083-6,ToDate,2026-06-13T23:59:59,{}
5,IT::UicOperatingPeriod:TRENITALIA:10-5A02-0083-6,ValidDayBits,1111111111111111111111111111100000000000011111...,{}


## Comment

The `UicOperatingPeriod` structure is now clear. Each element contains exactly
three fields:

- `FromDate` — the start date of the validity window for the bit string
- `ToDate` — the end date of the validity window
- `ValidDayBits` — a binary string where each position represents one day
  in the window, with 1 meaning the service runs and 0 meaning it does not

The two sampled periods show different validity windows:
- `10-5A01` runs from 2026-04-03 to 2026-05-23 — a short seasonal window
- `10-5A02` runs from 2025-12-14 to 2026-06-13 — the full feed validity window

The ID of each `UicOperatingPeriod` matches the corresponding `DayType` ID,
for example `IT::UicOperatingPeriod:TRENITALIA:10-5A01-0083-6` matches
`IT::DayType:TRENITALIA:10-5A01-0083-6`. This means the link between DayType
and UicOperatingPeriod can be established by ID rather than through
`DayTypeAssignment`, which simplifies the calendar extraction significantly.

The calendar reconstruction for Italy will follow the same bit string approach
as Luxembourg and France. This means linking the `ValidDayBits` string to the `FromDate`
and expand each 1 into an active date.

## Extract all UicOperatingPeriods

Now that the structure is confirmed, I extract all `UicOperatingPeriod` elements
with their `FromDate`, `ToDate` and `ValidDayBits` to build the full NeTEx
active date table for Italy.

In [24]:
def extract_all_uic_periods_gzip(gzip_path):
    rows = []
    current = None
    depth = 0
    events_scanned = 0

    with gzip.open(gzip_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            events_scanned += 1

            if events_scanned % 100000 == 0:
                print(f"  {events_scanned:,} events scanned — {len(rows)} periods extracted", end="\r")

            if event == "start" and tag == "UicOperatingPeriod":
                depth += 1
                if depth == 1:
                    current = {
                        "uic_period_id":  elem.attrib.get("id"),
                        "from_date":      None,
                        "to_date":        None,
                        "valid_day_bits": None
                    }

            elif event == "end" and tag == "UicOperatingPeriod":
                if depth == 1 and current is not None:
                    rows.append(current)
                    current = None
                depth -= 1
                elem.clear()

            elif event == "end" and current is not None and depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "FromDate" and current["from_date"] is None:
                    current["from_date"] = text
                elif tag == "ToDate" and current["to_date"] is None:
                    current["to_date"] = text
                elif tag == "ValidDayBits" and current["valid_day_bits"] is None:
                    current["valid_day_bits"] = text
                elem.clear()

            elif event == "end":
                elem.clear()

    print(f"\nDone — {len(rows)} UicOperatingPeriods extracted.")
    return pd.DataFrame(rows)

netex_uic_periods = extract_all_uic_periods_gzip(netex_path)

netex_uic_periods["from_date"] = pd.to_datetime(
    netex_uic_periods["from_date"], errors="coerce"
).dt.normalize()
netex_uic_periods["to_date"] = pd.to_datetime(
    netex_uic_periods["to_date"], errors="coerce"
).dt.normalize()

display(netex_uic_periods.head(10))

  14,200,000 events scanned — 33054 periods extracted
Done — 33054 UicOperatingPeriods extracted.


,uic_period_id,from_date,to_date,valid_day_bits
0,IT::UicOperatingPeriod:TRENITALIA:10-5A01-0083-6,2026-04-03,2026-05-23,1000000000000000000000000000000000000000010000...
1,IT::UicOperatingPeriod:TRENITALIA:10-5A02-0083-6,2025-12-14,2026-06-13,1111111111111111111111111111100000000000011111...
2,IT::UicOperatingPeriod:TRENITALIA:10-5A03-0083-6,2026-02-28,2026-03-01,11
3,IT::UicOperatingPeriod:TRENITALIA:10-5A04-0083-6,2026-01-12,2026-01-23,111111111111
4,IT::UicOperatingPeriod:TRENITALIA:10201-59EF-0...,2026-05-01,2026-06-13,11100000000000011000001100000110100011000001
5,IT::UicOperatingPeriod:TRENITALIA:10201-59F0-0...,2026-05-10,2026-05-10,1
6,IT::UicOperatingPeriod:TRENITALIA:10201-59F1-0...,2026-05-09,2026-05-09,1
7,IT::UicOperatingPeriod:TRENITALIA:10201-59F2-0...,2026-04-25,2026-04-26,11
8,IT::UicOperatingPeriod:TRENITALIA:10201-59F3-0...,2026-04-19,2026-04-19,1
9,IT::UicOperatingPeriod:TRENITALIA:10201-59F4-0...,2026-03-14,2026-04-18,110000010000000000000111000010000001


## Comment

All 33,054 `UicOperatingPeriod` elements extracted successfully. A few observations:

- `FromDate`, `ToDate` and `ValidDayBits` are present for all sampled rows
- Validity windows vary greatly. Some cover the full feed window (row 1:
  2025-12-14 to 2026-06-13) while others cover a single day (rows 5, 6, 7)
- Some bit strings are very short. For example "11" for a 2-day window, "1" for a single
  day, which is consistent and correct
- The full feed window runs from **2025-12-14 to 2026-06-13**, matching what
  was seen in the file header

The table is ready for the active date reconstruction. The next step is to
expand each `ValidDayBits` string into individual active dates using the
`FromDate` as the linking element, following the same approach used for Luxembourg
and France.

## Build NeTEx active dates from UicOperatingPeriod

Each `UicOperatingPeriod` contains a `ValidDayBits` bit string linked to a
`FromDate`. Each position in the string represents one calendar day starting
from `FromDate`, where `1` means the service runs and `0` means it does not.

This is the same approach used for Luxembourg and France. The bit string is
expanded into individual active dates and linked to the corresponding `DayType`
by replacing `UicOperatingPeriod` with `DayType` in the ID , since the IDs
are identical except for the element type prefix.

In [25]:
# Build NeTEx active dates from UicOperatingPeriod ValidDayBits
netex_active_date_rows = []
total = len(netex_uic_periods)

for i, (_, row) in enumerate(netex_uic_periods.iterrows(), start=1):
    if i % 1000 == 0 or i == total:
        print(f"  [{i}/{total}]", end="\r")

    bits      = row["valid_day_bits"]
    from_date = row["from_date"]

    if not bits or pd.isna(from_date):
        continue

    # Derive matching DayType ID by replacing UicOperatingPeriod with DayType
    daytype_ref = row["uic_period_id"].replace(
        "UicOperatingPeriod", "DayType"
    )

    for j, bit in enumerate(bits):
        if bit == "1":
            active_date = from_date + pd.Timedelta(days=j)
            netex_active_date_rows.append({
                "daytype_ref": daytype_ref,
                "active_date": active_date
            })

print(f"\nDone — {len(netex_active_date_rows)} active date rows built.")
netex_active_dates = pd.DataFrame(netex_active_date_rows)
display(netex_active_dates.head(10))

  [33054/33054]
Done — 1248576 active date rows built.


,daytype_ref,active_date
0,IT::DayType:TRENITALIA:10-5A01-0083-6,2026-04-03
1,IT::DayType:TRENITALIA:10-5A01-0083-6,2026-05-14
2,IT::DayType:TRENITALIA:10-5A01-0083-6,2026-05-23
3,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-14
4,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-15
5,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-16
6,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-17
7,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-18
8,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-19
9,IT::DayType:TRENITALIA:10-5A02-0083-6,2025-12-20


## Comment

The active date expansion completed successfully:

- **1,248,576 active date rows** built from 33,054 UicOperatingPeriods
- The DayType IDs were correctly derived by replacing `UicOperatingPeriod`
  with `DayType` in the ID string
- The sample shows dates spread across the feed window from 2025-12-14
  onwards, confirming the expansion worked correctly
- DayType `10-5A01` has only 3 active dates (April 3, May 14, May 23) 
  consistent with the short bit string seen earlier
- DayType `10-5A02` has consecutive daily entries starting from 2025-12-14
   consistent with the long bit string covering the full feed window

## Prepare NeTEx calendar summary

The active dates table is aggregated into one summary row per DayType,
showing the number of active days, the first and last active date. This
summary will be used in Part 3 for the calendar comparison.

In [26]:
# Prepare NeTEx calendar summary
netex_active_dates["active_date"] = pd.to_datetime(
    netex_active_dates["active_date"], errors="coerce"
).dt.normalize()

netex_calendar_summary = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .groupby("daytype_ref", as_index=False)
    .agg(
        n_active_days=("active_date", "nunique"),
        first_active_date=("active_date", "min"),
        last_active_date=("active_date", "max")
    )
    .sort_values("daytype_ref")
    .reset_index(drop=True)
)

netex_calendar_summary_quality = pd.DataFrame({
    "metric": [
        "calendar summary rows",
        "daytype_ref with only 1 active day",
        "missing first_active_date",
        "missing last_active_date",
        "earliest first_active_date",
        "latest last_active_date"
    ],
    "value": [
        len(netex_calendar_summary),
        netex_calendar_summary["n_active_days"].eq(1).sum(),
        netex_calendar_summary["first_active_date"].isna().sum(),
        netex_calendar_summary["last_active_date"].isna().sum(),
        netex_calendar_summary["first_active_date"].min(),
        netex_calendar_summary["last_active_date"].max()
    ]
})

display(netex_calendar_summary.head(10))
display(netex_calendar_summary_quality)

,daytype_ref,n_active_days,first_active_date,last_active_date
0,IT::DayType:TRENITALIA:10-5A01-0083-6,3,2026-04-03,2026-05-23
1,IT::DayType:TRENITALIA:10-5A02-0083-6,165,2025-12-14,2026-06-13
2,IT::DayType:TRENITALIA:10-5A03-0083-6,2,2026-02-28,2026-03-01
3,IT::DayType:TRENITALIA:10-5A04-0083-6,12,2026-01-12,2026-01-23
4,IT::DayType:TRENITALIA:10201-59EF-0083-1,13,2026-05-01,2026-06-13
5,IT::DayType:TRENITALIA:10201-59F0-0083-1,1,2026-05-10,2026-05-10
6,IT::DayType:TRENITALIA:10201-59F1-0083-1,1,2026-05-09,2026-05-09
7,IT::DayType:TRENITALIA:10201-59F2-0083-1,2,2026-04-25,2026-04-26
8,IT::DayType:TRENITALIA:10201-59F3-0083-1,1,2026-04-19,2026-04-19
9,IT::DayType:TRENITALIA:10201-59F4-0083-1,8,2026-03-14,2026-04-18


,metric,value
0,calendar summary rows,33054
1,daytype_ref with only 1 active day,5072
2,missing first_active_date,0
3,missing last_active_date,0
4,earliest first_active_date,2025-12-14 00:00:00
5,latest last_active_date,2026-06-13 00:00:00


## Comment

The NeTEx calendar summary was built successfully with **33,054 DayType rows**,
matching exactly the number of UicOperatingPeriods extracted:

- No missing dates, all DayTypes have a valid first and last active date
- **5,072 DayTypes with only 1 active day** — about 15% of all DayTypes
  represent a single specific service day, likely special or seasonal trips
- The feed validity window runs from **2025-12-14 to 2026-06-13** — about
  6 months, significantly shorter than the GTFS feed window of 332 days
- Active day counts vary widely from 1 day to 165 days, reflecting the
  mix of regular and occasional services in the Trenitalia network


## Summary of NeTEx exploration

**File structure:**
- Single gzip-compressed XML file 
- Single operator: Trenitalia

**Stops:**
- 1,910 StopPlaces extracted, all with names and coordinates
- Flat structure, one Quay per StopPlace, no parent-child hierarchy
- IDs follow `IT::StopPlace:otherTRENITALIA:...` format, no shared IDs
  with GTFS, so coordinate proximity matching will be used

**Lines:**
- 15 Lines extracted: 14 rail, 1 bus
- Lines represent **service categories** (Regionale, Frecciarossa, Intercity)
  rather than geographic routes
- All lines have a `PublicCode`but these short codes (REG, FR, IC) are
  not comparable to GTFS route long names (Firenze - Pisa - Livorno)

**Calendar:**
- Uses `UicOperatingPeriod` with `ValidDayBits` bit strings 
- 33,054 DayTypes with a validity window from 2025-12-14 to 2026-06-13
- 5,072 DayTypes with only 1 active day, many single-day special services

# Part 3: GTFS and NeTEx comparison

## 3.1 Stop-level comparison


## Note on geographic scope for the stop comparison

The GTFS feed covers Trenitalia services in Tuscany and neighbouring regions,
while the NeTEx feed covers the full national Trenitalia network. Using all
1,910 NeTEx StopPlaces for the comparison would make the NeTEx-side coverage
rate artificially low since most NeTEx stops are in regions not covered by GTFS
at all, so they would never be matched by definition.

To make the comparison fairer, the NeTEx StopPlaces are first filtered to the
geographic area covered by the GTFS feed. A small buffer of 0.5 degrees is
added around the GTFS bounding box to avoid excluding stops that are just
outside the boundary but still relevant for the comparison.

The GTFS-side match rate is not affected by this filter. It only improves
the meaningfulness of the NeTEx-side coverage rate.

## Restrict the geographic area

The following code cell finds the minimum and maximum latitude and longitude across all GTFS stops. This defines a rectangle that covers the entire geographic area of the GTFS feed. Then it filters NeTEx StopPlaces to only keep those that fall inside that rectangle (plus a small 0.5° buffer on each side). Any NeTEx stop outside this rectangle is in a region the GTFS feed doesn't cover at all, so it will not be considered in the comparison.

In [27]:
# Define the geographic bounding box of the GTFS feed
gtfs_lat_min = gtfs_stations["gtfs_lat"].min()
gtfs_lat_max = gtfs_stations["gtfs_lat"].max()
gtfs_lon_min = gtfs_stations["gtfs_lon"].min()
gtfs_lon_max = gtfs_stations["gtfs_lon"].max()

print(f"GTFS bounding box:")
print(f"  Latitude:  {gtfs_lat_min:.4f} to {gtfs_lat_max:.4f}")
print(f"  Longitude: {gtfs_lon_min:.4f} to {gtfs_lon_max:.4f}")

# Add a small buffer of 0.5 degrees around the GTFS area
buffer = 0.5
netex_in_gtfs_area = netex_stopplaces[
    (netex_stopplaces["netex_lat"] >= gtfs_lat_min - buffer) &
    (netex_stopplaces["netex_lat"] <= gtfs_lat_max + buffer) &
    (netex_stopplaces["netex_lon"] >= gtfs_lon_min - buffer) &
    (netex_stopplaces["netex_lon"] <= gtfs_lon_max + buffer)
].copy()

print(f"\nNeTEx StopPlaces in GTFS area (+{buffer}° buffer): {len(netex_in_gtfs_area)}")
print(f"NeTEx StopPlaces outside GTFS area: {len(netex_stopplaces) - len(netex_in_gtfs_area)}")

GTFS bounding box:
  Latitude:  41.8724 to 45.4863
  Longitude: 8.4702 to 13.4976

NeTEx StopPlaces in GTFS area (+0.5° buffer): 1040
NeTEx StopPlaces outside GTFS area: 870


## Comment

The GTFS bounding box covers a large area of central and northern Italy 
from latitude 41.87°N to 45.49°N and longitude 8.47°E to 13.50°E. This
confirms that the feed extends well beyond Tuscany into Lombardy, Liguria
and Emilia-Romagna due to Trenitalia's inter-regional services.

Applying the 0.5° buffer reduces the NeTEx comparison set from 1,910 to
**1,040 StopPlaces** just over half the national network. The remaining
870 StopPlaces are in southern Italy and other regions not covered by the
GTFS feed at all, and are excluded from the comparison.

## Stop-level comparison: GTFS vs NeTEx

The coordinate proximity comparison is now run between the 313 GTFS stops
and the 1,040 NeTEx StopPlaces within the GTFS geographic area. A match
is accepted if the nearest NeTEx StopPlace is within 100 metres.

In [28]:
# Use geographically filtered NeTEx stops
gtfs_valid  = gtfs_stations.dropna(subset=["gtfs_lat", "gtfs_lon"]).copy()
netex_valid = netex_in_gtfs_area.dropna(subset=["netex_lat", "netex_lon"]).copy()

# Prepare coordinate arrays in radians
gtfs_coords_rad  = np.radians(gtfs_valid[["gtfs_lat", "gtfs_lon"]].values)
netex_coords_rad = np.radians(netex_valid[["netex_lat", "netex_lon"]].values)

# Build BallTree from NeTEx coordinates
tree = BallTree(netex_coords_rad, metric="haversine")

# Find nearest NeTEx StopPlace for each GTFS stop
distances_rad, indices = tree.query(gtfs_coords_rad, k=1)

# Convert distance from radians to metres
earth_radius_m  = 6371000
distances_m     = distances_rad[:, 0] * earth_radius_m
nearest_indices = indices[:, 0]

# Attach results to GTFS stops
gtfs_valid = gtfs_valid.reset_index(drop=True)
nearest_netex = netex_valid.iloc[nearest_indices].reset_index(drop=True)

gtfs_valid["nearest_netex_id"]   = nearest_netex["netex_stopplace_id"].values
gtfs_valid["nearest_netex_name"] = nearest_netex["netex_stopplace_name"].values
gtfs_valid["nearest_netex_lat"]  = nearest_netex["netex_lat"].values
gtfs_valid["nearest_netex_lon"]  = nearest_netex["netex_lon"].values
gtfs_valid["distance_m"]         = distances_m

# Apply 100m threshold
THRESHOLD_M = 100
gtfs_valid["matched"] = gtfs_valid["distance_m"] <= THRESHOLD_M

# Summary
stop_match_summary = pd.DataFrame({
    "metric": [
        "GTFS stops",
        "NeTEx StopPlaces in GTFS area",
        "GTFS stops matched (within 100m)",
        "GTFS stops unmatched",
        "GTFS-side match rate (%)",
        "NeTEx-side coverage rate (%)"
    ],
    "value": [
        len(gtfs_valid),
        len(netex_valid),
        gtfs_valid["matched"].sum(),
        (~gtfs_valid["matched"]).sum(),
        round(gtfs_valid["matched"].mean() * 100, 2),
        round(gtfs_valid["matched"].sum() / len(netex_valid) * 100, 2)
    ]
})
display(stop_match_summary)

print("Example matched stops:")
display(gtfs_valid[gtfs_valid["matched"]][
    ["gtfs_stop_id", "gtfs_stop_name", "nearest_netex_id",
     "nearest_netex_name", "distance_m"]
].head(10))

print("Example unmatched stops:")
display(gtfs_valid[~gtfs_valid["matched"]].sort_values(
    "distance_m", ascending=False
)[["gtfs_stop_id", "gtfs_stop_name", "nearest_netex_id",
   "nearest_netex_name", "distance_m"]].head(10))

,metric,value
0,GTFS stops,313.00
1,NeTEx StopPlaces in GTFS area,1040.00
2,GTFS stops matched (within 100m),276.00
3,GTFS stops unmatched,37.00
4,GTFS-side match rate (%),88.18
5,NeTEx-side coverage rate (%),26.54


Example matched stops:


,gtfs_stop_id,gtfs_stop_name,nearest_netex_id,nearest_netex_name,distance_m
0,S01645_1,Milano P. Garibaldi,IT::StopPlace:railTRENITALIA:830001645,MILANO PORTA GARIBALDI,14.833047
1,S01700_1,Milano C.Le,IT::StopPlace:otherTRENITALIA:830001700,MILANO CENTRALE,15.973767
2,S01701_1,Milano Lambrate,IT::StopPlace:railTRENITALIA:830001701,MILANO LAMBRATE,10.780562
3,S01820_1,Milano Rogoredo,IT::StopPlace:otherTRENITALIA:830001820,MILANO ROGOREDO,10.179170
4,S01825_1,Lodi,IT::StopPlace:otherTRENITALIA:830001825,LODI,20.980496
5,S01827_1,Casalpusterlengo,IT::StopPlace:railTRENITALIA:830001827,CASALPUSTERLENGO,11.415757
6,S01828_1,Codogno,IT::StopPlace:railTRENITALIA:830001828,CODOGNO,14.010552
7,S04220_1,Genova Sampierdarena,IT::StopPlace:otherTRENITALIA:830004220,GENOVA SAMPIERDARENA,8.382604
8,S04529_1,Albisola,IT::StopPlace:otherTRENITALIA:830004529,ALBISOLA,13.077184
9,S04530_1,Celle,IT::StopPlace:otherTRENITALIA:830004530,CELLE,14.458447


Example unmatched stops:


,gtfs_stop_id,gtfs_stop_name,nearest_netex_id,nearest_netex_name,distance_m
239,S06851_1,Rigomagno,IT::StopPlace:railTRENITALIA:830006852,SINALUNGA,5488.393394
74,S05138_1,Rastignano,IT::StopPlace:railTRENITALIA:830005130,BOLOGNA S.RUFFILLO,2472.128696
243,S06867_1,Siena Zona Ind.,IT::StopPlace:railTRENITALIA:830006869,PONTE A TRESSA,2235.852386
193,S06601_1,Montorsoli,IT::StopPlace:railTRENITALIA:830006953,FIESOLE CALDINE,2089.217283
200,S06612_1,Popolano,IT::StopPlace:otherTRENITALIA:830006620,POPOLANO,1444.493036
102,S06040_1,Viareggio,IT::StopPlace:otherTRENITALIA:830006040,VIAREGGIO,982.933193
113,S06082_1,Subbiano,IT::StopPlace:railTRENITALIA:830034103,Subbiano,968.785195
108,S06074_1,Poppi,IT::StopPlace:railTRENITALIA:830034106,Poppi,876.179664
75,S05139_1,Rastignano Sfm,IT::StopPlace:railTRENITALIA:830005139,Musiano-Pian di Macina,852.034373
45,S04736_1,Riomaggiore,IT::StopPlace:railTRENITALIA:830004736,RIOMAGGIORE,786.533043


## Comment

The coordinate-based stop comparison gives a strong but not perfect result:

- **276 out of 313 GTFS stops matched** a NeTEx StopPlace within 100 metres,
  giving a **GTFS-side match rate of 88.18%**
- **37 GTFS stops were unmatched** , their nearest NeTEx StopPlace is more
  than 100 metres away
- The **NeTEx-side coverage rate is 26.54%** — low but expected, since the
  1,040 NeTEx StopPlaces in the area cover the full Trenitalia network while
  GTFS only covers 313 stops from the Tuscany regional feed

The matched stops show very small distances like for example Milano Porta Garibaldi matched
at 14.8m, Milano Lambrate at 10.8m, Genova Sampierdarena at 8.4m. These are
effectively exact matches, just with slightly different coordinate precision
between the two formats.

The unmatched stops reveal two patterns:
- Some are genuine mismatches like **Rigomagno** is 5.5 km from its nearest
  NeTEx stop, suggesting it exists in GTFS but not in the NeTEx national
  network
- Others are very close but just over the 100m threshold like **Viareggio**
  at 983m and **Riomaggiore** at 786m suggest a coordinate precision issue
  or slightly different station entrance points between the two formats

Overall the stop-level MVD is **partially confirmed** for Italy. The majority
of GTFS stops have a matching NeTEx StopPlace nearby, but the geographic scope
mismatch between the two feeds limits the completeness of the comparison.

## 3.2 Line-level comparison

The line comparison for Italy is structurally different from all other countries
analysed so far. GTFS models lines as **geographic routes** with origin-destination
names like "Firenze - Pisa - Livorno", while NeTEx models lines as **service
categories** with short public codes like "REG" or "FR". These two labelling
systems are not directly comparable by label matching.

The comparison is done in two ways:
- A direct label match to confirm the mismatch
- A transport mode comparison to check whether both formats at least agree
  on the type of service

In [29]:
# Prepare label sets
gtfs_label_set  = set(gtfs_lines["gtfs_line_label"].dropna().astype(str).str.strip())
netex_label_set = set(netex_lines["netex_public_code"].dropna().astype(str).str.strip())

matched_labels    = gtfs_label_set & netex_label_set
gtfs_only_labels  = gtfs_label_set - netex_label_set
netex_only_labels = netex_label_set - gtfs_label_set

line_match_summary = pd.DataFrame({
    "metric": [
        "GTFS unique line labels",
        "NeTEx unique public codes",
        "matched labels",
        "GTFS-only labels",
        "NeTEx-only labels",
        "GTFS-side match rate (%)",
        "NeTEx-side match rate (%)"
    ],
    "value": [
        len(gtfs_label_set),
        len(netex_label_set),
        len(matched_labels),
        len(gtfs_only_labels),
        len(netex_only_labels),
        round(len(matched_labels) / len(gtfs_label_set)  * 100, 2) if gtfs_label_set  else 0.0,
        round(len(matched_labels) / len(netex_label_set) * 100, 2) if netex_label_set else 0.0
    ]
})
display(line_match_summary)

print("GTFS line labels:")
display(pd.DataFrame({"gtfs_label": sorted(gtfs_label_set)}))

print("NeTEx public codes:")
display(pd.DataFrame({"netex_code": sorted(netex_label_set)}))

,metric,value
0,GTFS unique line labels,14.0
1,NeTEx unique public codes,15.0
2,matched labels,0.0
3,GTFS-only labels,14.0
4,NeTEx-only labels,15.0
5,GTFS-side match rate (%),0.0
6,NeTEx-side match rate (%),0.0


GTFS line labels:


,gtfs_label
0,(Fi) Prato - Bologna
1,(Ge) La Spezia - Pisa
2,Firenze - Arezzo - Chiusi (Rm)
3,Firenze - Borgo Sl - Faenza
4,Firenze - Borgo Sl Via Pontassieve
5,Firenze - Empoli - Siena
6,Firenze - Pisa - Livorno
7,Firenze - Pistoia - Lucca - Viareggio
8,La Spezia - Parma (Mi)(Bg)
9,Pisa - Livorno - Grosseto (Rm)


NeTEx public codes:


,netex_code
0,BUS
1,EC
2,EN
3,EXP
4,FA
5,FB
6,FL
7,FR
8,IC
9,ICN


## Comment

The line comparison gives a **0% match rate on both sides**. None of the 14
GTFS line labels match any of the 15 NeTEx public codes. This is not a data
quality issue but a fundamental structural difference in how the two formats
model lines for Trenitalia:

- **GTFS** models lines as geographic origin-destination routes, for example
  "Firenze - Pisa - Livorno" or "Siena - Chiusi". Each line represents a
  specific physical route through the network
- **NeTEx** models lines as service categories, for example "REG" (Regionale),
  "FR" (Frecciarossa), "IC" (Intercity). Each line represents a type of service
  that can run on many different routes

These two approaches are conceptually incompatible. A geographic route in GTFS
can be served by multiple NeTEx service categories, and a NeTEx service category
can cover many different GTFS routes. No label matching is possible between them.

This is a significant finding for the thesis: **Italy demonstrates a case where
GTFS and NeTEx both contain line information but organise it in fundamentally
different ways**, making a direct comparison impossible. The line-level MVD
cannot be confirmed for Italy using label matching alone.

## Attempt to improve line matching by keyword and transport mode

Since the direct label comparison gave a 0% match rate, I tried to improve
the result in two ways:

- Comparing transport modes between the two formats to check at least mode-level
  consistency
- Searching for NeTEx service type keywords (such as "Regionale", "Intercity",
  "Frecciarossa") inside the GTFS route long names to see if any automatic
  mapping was possible

**Transport mode comparison** showed that GTFS uses only "rail" while NeTEx
uses "rail" and "bus" so the two formats agree on the dominant mode.

**Keyword matching** returned no results. All GTFS route names are purely
geographic origin-destination strings with no service type keywords. There is
no automatic way to map "Firenze - Pisa - Livorno" to a NeTEx category like
"REG" or "FR" without external knowledge of which service type operates each
route.

The keyword matching code is kept here for transparency but produces no useful
output. The 0% line label match rate stands as the result for Italy, and is
documented as a structural finding rather than a data quality issue.

In [30]:
# Compare transport modes between GTFS and NeTEx
gtfs_modes  = set(gtfs_lines["gtfs_route_type"].map({2: "rail"}).dropna())
netex_modes = set(netex_lines["netex_transport_mode"].dropna())

print("GTFS transport modes:", gtfs_modes)
print("NeTEx transport modes:", netex_modes)

# Check if any GTFS route long names contain NeTEx service keywords
netex_keywords = {
    "REG":  ["Regionale", "regionale"],
    "RV":   ["Regionale Veloce", "veloce"],
    "IC":   ["Intercity", "intercity"],
    "FR":   ["Frecciarossa", "frecciarossa"],
    "FA":   ["Frecciargento", "frecciargento"],
    "FB":   ["Frecciabianca", "frecciabianca"],
    "EC":   ["Eurocity", "eurocity"],
    "EN":   ["Euronight", "euronight"],
    "ICN":  ["InterCityNotte", "notte"],
    "EXP":  ["Espresso", "espresso"],
    "BUS":  ["Autobus", "bus", "Bus"]
}

gtfs_lines["possible_netex_code"] = None
for code, keywords in netex_keywords.items():
    for keyword in keywords:
        mask = gtfs_lines["gtfs_line_label"].str.contains(keyword, case=False, na=False)
        gtfs_lines.loc[mask, "possible_netex_code"] = code

display(gtfs_lines[["gtfs_line_label", "gtfs_route_type", "possible_netex_code"]])

GTFS transport modes: {'rail'}
NeTEx transport modes: {'rail', 'bus'}


,gtfs_line_label,gtfs_route_type,possible_netex_code
0,Firenze - Pisa - Livorno,2,None
1,(Ge) La Spezia - Pisa,2,None
2,Siena - Chiusi,2,None
3,(Fi) Prato - Bologna,2,None
4,Firenze - Borgo Sl Via Pontassieve,2,None
5,Pisa - Livorno - Grosseto (Rm),2,None
6,Firenze - Arezzo - Chiusi (Rm),2,None
7,Firenze - Pistoia - Lucca - Viareggio,2,None
8,Siena - Grosseto,2,None
9,Pisa - Lucca - Aulla,2,None


## 3.3 Calendar comparison

GTFS uses explicit date entries in `calendar_dates.txt`, while NeTEx uses
`ValidDayBits` strings inside `UicOperatingPeriod` elements. Both are
converted into bit strings so they can be compared directly.

Since the two formats use completely different ID systems, a direct ID match
is not possible. The comparison is done at the **pattern level** over a shared
date window: each service's active dates are encoded as a binary string, and
two patterns are considered a match only if that string is identical on both
sides.

In [31]:
# Build GTFS dates dictionary
gtfs_dates_by_id = {
    service_id: set(pd.to_datetime(list(dates)).normalize())
    for service_id, dates in gtfs_active_dates.items()
}

# Build NeTEx dates dictionary
netex_dates_by_id = (
    netex_active_dates
    .dropna(subset=["daytype_ref", "active_date"])
    .assign(active_date=lambda df: pd.to_datetime(df["active_date"]).dt.normalize())
    .groupby("daytype_ref")["active_date"]
    .apply(set)
    .to_dict()
)

# Define shared comparison window
all_gtfs_dates  = [d for dates in gtfs_dates_by_id.values()  for d in dates]
all_netex_dates = [d for dates in netex_dates_by_id.values() for d in dates]

gtfs_min_date  = min(all_gtfs_dates)
gtfs_max_date  = max(all_gtfs_dates)
netex_min_date = min(all_netex_dates)
netex_max_date = max(all_netex_dates)

shared_start_date = max(gtfs_min_date, netex_min_date)
shared_end_date   = min(gtfs_max_date, netex_max_date)
shared_dates      = pd.date_range(shared_start_date, shared_end_date, freq="D")

shared_window_summary = pd.DataFrame({
    "metric": [
        "GTFS earliest active date",
        "GTFS latest active date",
        "NeTEx earliest active date",
        "NeTEx latest active date",
        "shared start date",
        "shared end date",
        "number of days in shared window"
    ],
    "value": [
        gtfs_min_date,
        gtfs_max_date,
        netex_min_date,
        netex_max_date,
        shared_start_date,
        shared_end_date,
        len(shared_dates)
    ]
})
display(shared_window_summary)

,metric,value
0,GTFS earliest active date,2026-01-15 00:00:00
1,GTFS latest active date,2026-12-12 00:00:00
2,NeTEx earliest active date,2025-12-14 00:00:00
3,NeTEx latest active date,2026-06-13 00:00:00
4,shared start date,2026-01-15 00:00:00
5,shared end date,2026-06-13 00:00:00
6,number of days in shared window,150


## Comment

The shared date window runs from **2026-01-15 to 2026-06-13**, covering **150 days**:

- The GTFS feed covers almost a full year (2026-01-15 to 2026-12-12)
- The NeTEx feed covers about 6 months (2025-12-14 to 2026-06-13)
- The shared window is limited by the NeTEx end date on one side and the
  GTFS start date on the other
- 150 days is a reasonable window for a pattern comparison. It covers
  winter, spring and early summer services including seasonal variations



## Build and compare calendar activity patterns

Each service's active dates are encoded as a binary activity pattern over
the shared date window. Two patterns are considered equivalent if they are
identical, meaning both services run on exactly the same days within the
150-day window. The match rate is reported from both sides.

In [32]:
# Build activity patterns
def build_activity_pattern(active_dates, comparison_dates):
    active_dates = set(active_dates)
    return "".join("1" if d in active_dates else "0" for d in comparison_dates)

# Build GTFS patterns
gtfs_pattern_rows = []
total_gtfs = len(gtfs_dates_by_id)
for i, (service_id, active_dates) in enumerate(gtfs_dates_by_id.items(), start=1):
    print(f"  GTFS [{i}/{total_gtfs}]", end="\r")
    window_dates = {d for d in active_dates if shared_start_date <= d <= shared_end_date}
    gtfs_pattern_rows.append({
        "gtfs_service_id":          service_id,
        "n_active_days_window":     len(window_dates),
        "first_active_date_window": min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window":  max(window_dates) if window_dates else pd.NaT,
        "activity_pattern":         build_activity_pattern(window_dates, shared_dates)
    })
print(f"\nGTFS done — {total_gtfs} calendars processed.")
gtfs_calendar_patterns = pd.DataFrame(gtfs_pattern_rows)

# Build NeTEx patterns
netex_pattern_rows = []
total_netex = len(netex_dates_by_id)
for i, (daytype_ref, active_dates) in enumerate(netex_dates_by_id.items(), start=1):
    print(f"  NeTEx [{i}/{total_netex}]", end="\r")
    window_dates = {d for d in active_dates if shared_start_date <= d <= shared_end_date}
    netex_pattern_rows.append({
        "netex_daytype_ref":         daytype_ref,
        "n_active_days_window":      len(window_dates),
        "first_active_date_window":  min(window_dates) if window_dates else pd.NaT,
        "last_active_date_window":   max(window_dates) if window_dates else pd.NaT,
        "activity_pattern":          build_activity_pattern(window_dates, shared_dates)
    })
print(f"\nNeTEx done — {total_netex} calendars processed.")
netex_calendar_patterns = pd.DataFrame(netex_pattern_rows)

# Filter to active patterns only
gtfs_calendar_patterns_active = gtfs_calendar_patterns[
    gtfs_calendar_patterns["n_active_days_window"] > 0
].copy()
netex_calendar_patterns_active = netex_calendar_patterns[
    netex_calendar_patterns["n_active_days_window"] > 0
].copy()

# Compare unique patterns
gtfs_pattern_set   = set(gtfs_calendar_patterns_active["activity_pattern"])
netex_pattern_set  = set(netex_calendar_patterns_active["activity_pattern"])
matched_patterns   = gtfs_pattern_set & netex_pattern_set
gtfs_only_patterns = gtfs_pattern_set - netex_pattern_set
netex_only_patterns = netex_pattern_set - gtfs_pattern_set

pattern_match_summary = pd.DataFrame({
    "metric": [
        "GTFS calendar rows",
        "GTFS rows with active days in shared window",
        "GTFS unique activity patterns",
        "NeTEx calendar rows",
        "NeTEx rows with active days in shared window",
        "NeTEx unique activity patterns",
        "matched activity patterns",
        "GTFS-side pattern match rate (%)",
        "NeTEx-side pattern match rate (%)"
    ],
    "value": [
        len(gtfs_calendar_patterns),
        len(gtfs_calendar_patterns_active),
        len(gtfs_pattern_set),
        len(netex_calendar_patterns),
        len(netex_calendar_patterns_active),
        len(netex_pattern_set),
        len(matched_patterns),
        round(len(matched_patterns) / len(gtfs_pattern_set)  * 100, 2) if gtfs_pattern_set  else 0.0,
        round(len(matched_patterns) / len(netex_pattern_set) * 100, 2) if netex_pattern_set else 0.0
    ]
})
display(pattern_match_summary)

  GTFS [231/231]
GTFS done — 231 calendars processed.
  NeTEx [33054/33054]
NeTEx done — 33054 calendars processed.


,metric,value
0,GTFS calendar rows,231.00
1,GTFS rows with active days in shared window,221.00
2,GTFS unique activity patterns,87.00
3,NeTEx calendar rows,33054.00
4,NeTEx rows with active days in shared window,31507.00
5,NeTEx unique activity patterns,5431.00
6,matched activity patterns,44.00
7,GTFS-side pattern match rate (%),50.57
8,NeTEx-side pattern match rate (%),0.81


## Comment

The pattern-level calendar comparison gives a mixed result:

- **GTFS produces 87 unique patterns** from 221 active service IDs in the
  shared window. A relatively small number, suggesting many services share
  the same calendar pattern
- **NeTEx produces 5,431 unique patterns** from 31,507 active DayTypes.
  Much more granular, with one pattern per individual trip rather than per
  service group
- **44 patterns matched**, giving a **GTFS-side rate of 50.57%** and a very
  low **NeTEx-side rate of 0.81%**

The asymmetry is explained by the same granularity difference seen in Sweden.
NeTEx models calendar at the individual trip level (33,054 DayTypes for
33,054 ServiceJourneys) while GTFS groups trips into 231 service IDs. This
means GTFS patterns are broad and reused across many trips, while NeTEx
patterns are highly specific and rarely repeated.

The 50.57% GTFS-side rate is a reasonable result. About half of GTFS
calendar patterns have an exact equivalent in NeTEx. The low NeTEx-side
rate reflects the granularity difference rather than missing calendar data.

## Calendar comparison: conclusion for Italy

The calendar comparison gives a partial result. Both formats contain the
information needed to reconstruct active service dates, but they organise
it at very different levels of granularity. A summary of the findings is
presented below.

In [33]:
# Calendar comparison summary table
calendar_conclusion = pd.DataFrame({
    "metric": [
        "shared date window",
        "number of days in window",
        "GTFS service IDs with active dates",
        "GTFS unique patterns",
        "NeTEx DayTypes with active dates",
        "NeTEx unique patterns",
        "matched patterns",
        "GTFS-side pattern match rate (%)",
        "NeTEx-side pattern match rate (%)"
    ],
    "value": [
        f"{shared_start_date.date()} to {shared_end_date.date()}",
        len(shared_dates),
        len(gtfs_calendar_patterns_active),
        len(gtfs_pattern_set),
        len(netex_calendar_patterns_active),
        len(netex_pattern_set),
        len(matched_patterns),
        round(len(matched_patterns) / len(gtfs_pattern_set)  * 100, 2),
        round(len(matched_patterns) / len(netex_pattern_set) * 100, 2)
    ]
})
display(calendar_conclusion)

,metric,value
0,shared date window,2026-01-15 to 2026-06-13
1,number of days in window,150
2,GTFS service IDs with active dates,221
3,GTFS unique patterns,87
4,NeTEx DayTypes with active dates,31507
5,NeTEx unique patterns,5431
6,matched patterns,44
7,GTFS-side pattern match rate (%),50.57
8,NeTEx-side pattern match rate (%),0.81


## Comment

The summary confirms the key findings of the Italian calendar comparison:

- The **150-day shared window** covers the overlap between the two feeds
- GTFS has **221 active service IDs** producing only **87 unique patterns** —
  on average each pattern is shared by about 2.5 service IDs, confirming that
  GTFS groups trips under shared service calendars
- NeTEx has **31,507 active DayTypes** producing **5,431 unique patterns** —
  one DayType per ServiceJourney, meaning every individual trip has its own
  calendar entry
- **44 patterns matched**, giving a GTFS-side rate of **50.57%** and a
  NeTEx-side rate of **0.81%**

The calendar-level MVD is **partially confirmed** for Italy. Both formats
contain the data needed to reconstruct when services run, but the structural
difference in granularity, GTFS grouping trips into service IDs vs. NeTEx
assigning one DayType per trip, makes a direct comparison difficult and
produces an inherently asymmetric result.

## Overall summary

**Stops — partially confirmed**
- 88.18% of GTFS stops matched a NeTEx StopPlace within 100 metres
- Comparison restricted to the geographic area of the GTFS feed
- Unmatched stops are either genuinely absent from NeTEx or have small
  coordinate differences between the two formats
- NeTEx-side coverage is low (26.54%) due to the scope mismatch. NeTEx
  covers the full national Trenitalia network while GTFS covers only the
  Tuscany regional subset

**Lines — not confirmed**
- 0% match rate on both sides, the two formats use fundamentally different
  labelling systems
- GTFS uses geographic origin-destination names (Firenze - Pisa - Livorno)
- NeTEx uses service category codes (REG, FR, IC)
- Both formats contain line information but organise it in incompatible ways

**Calendar — partially confirmed**
- 50.57% GTFS-side pattern match rate over a 150-day shared window
- NeTEx models calendar at individual trip level (one DayType per trip)
  while GTFS groups trips into shared service IDs causing a large
  granularity difference
- Both formats contain the data needed to reconstruct active dates

**Overall**
- Italy is the most challenging case analysed so far, the scope mismatch
  between GTFS (regional) and NeTEx (national) affects all three comparison
  levels
- The line comparison revealed a fundamental structural difference in how
  the two formats model lines, which is a significant thesis finding
- The stop and calendar MVD are partially confirmed, the line MVD is not